# Tutorial of PLAN for continous waveform

This notebook demonstrates the use of PLAN for performing the earthquake signal detection and phase (P & S) picking, association and location on continuous data. Here, we demonstrates the results using Ridgecrest squence.

In [72]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [73]:
import sys   
sys.path.append("PLAN/")

# 1. import Lib 

In [74]:
import sys   
sys.path.append("..")
from utils.utils import *
from utils.detect_peaks import *
from utils.model_ridgecrest_vision import *
from utils.continue_dataloader import *

import seisbench.data as sbd
import seisbench.generate as sbg
from utils.seisbench_adapter import construct_plan_dataloader_from_seisbench
import pandas as pd
import numpy as np
import h5py


In [75]:
setup_seed(20)

In [28]:

def export_station_file_from_stead(meta: pd.DataFrame, out_path: str) -> pd.DataFrame:
    """
    Create a PLAN-style station file from STEAD metadata.
    """
    station_df = (
        meta[
            [
                "network_code",
                "receiver_code",
                "receiver_latitude",
                "receiver_longitude",
                "receiver_elevation_m",
            ]
        ]
        .drop_duplicates()
        .copy()
    )

    station_df.columns = ["#Network", "Station", "Latitude", "Longitude", "Elevation"]
    station_df["Sitename"] = ""
    station_df["StartTime"] = "1900-01-01T00:00:00"
    station_df["EndTime"] = "3000-01-01T00:00:00"

    station_df.to_csv(out_path, sep="|", index=False)
    return station_df

def get_waveforms(trace_ids):
    waveforms = []
    with h5py.File("data/chunk4.hdf5", "r") as h5:
        for trace_id in trace_ids:
            #ENZ -> ZEN
            x = np.asarray(h5['data'][trace_id])
            x = x[:, np.argsort([2, 1, 0])]
            x = x.transpose(1, 0)
            waveforms.append(x)
        return np.stack(waveforms)
    

# 2. import Data and construct DataLoader

In [ ]:
# We utilize 16 stations in CI Network for evaluation the performance of PLAN in Ridgecrest Squence Detection.
station_file_path = "PLAN/data/gmap-stations.txt"
station_pandas = pd.read_csv(station_file_path, sep='|')
# station_pandas = station_pandas.drop([0])
station_pandas.head(3)

,#Network,Station,Latitude,Longitude,Elevation,Sitename,StartTime,EndTime
1,CI,CCC,35.52495,-117.36453,670.0,Christmas Canyon China Lake,2001-06-22T00:00:00,3000-01-01T00:00:00
2,CI,CLC,35.81574,-117.59751,775.0,China Lake,1948-07-08T00:00:00,3000-01-01T00:00:00
3,CI,DAW,36.27148,-117.59214,1477.4,Darwin,2017-02-27T00:00:00,3000-01-01T00:00:00


In [78]:
station_file_path

'PLAN/data/gmap-stations.txt'

In [79]:
data_file = 'PLAN/data/1h_data/20190704173000.CI.'
inputdata = load_continous_data(station_file_path,data_file,data_length = 3600)
print(inputdata.shape)

16it [00:00, 70.02it/s]

(16, 3, 360001)


In [131]:
meta = pd.read_csv('data/chunk4.csv', sep=',')
meta['trace_start_time'] = pd.to_datetime(meta['trace_start_time'], format='mixed', errors='coerce')
meta_tmp = meta[meta.source_id == '606953116'].drop_duplicates(subset=[
                "network_code",
                "receiver_code",
                "receiver_latitude",
                "receiver_longitude",
                "receiver_elevation_m",
            ])
start_time = meta_tmp.trace_start_time.min()
station_pandas = export_station_file_from_stead(meta_tmp, 'PLAN/data/gmap-stations2.txt')
inputdata = get_waveforms(meta_tmp.trace_name.to_list())
print(inputdata.shape, station_pandas.shape)

(10, 3, 6000) (10, 8)


/tmp/ipykernel_11620/1525958688.py:1: DtypeWarning: Columns (0: source_mechanism_strike_dip_rake) have mixed types. Specify dtype option on import or set low_memory=False.
  meta = pd.read_csv('data/chunk4.csv', sep=',')


In [104]:
station_file_path = "PLAN/data/gmap-stations2.txt"

Define Dataloader

In [105]:
# start_time means how long after 17:30:00. Since the data is 100Hz sample, 30000 means 5 minute (5*60*100)
# end_time means how long after 17:30:00.
# interval means shift window when deal with continous data. (Here is 5 seconds)
# Here test start_time is 17:35:00, end_time is 17:41:20
ridge_loader,batch_start_time = construct_dataloader(inputdata,station_file_path,
                    #  start_time = 30000,
                    #  end_time = 70000,
                     start_time = 0,
                     end_time = inputdata.shape[-1],
                     interval = 500,
                     batchsize = 1,
                     num_workers = 0)

loader = list(ridge_loader)
len(loader)

12

# 3. Load Model

In [54]:
# if you have gpu, you can change this sentence as device = torch.device('cuda')
device = torch.device('cuda')
model_PLAN = Main_GCNN('Trans').to(device)
load_model_name = 'PLAN/model/model_PLAN_Ridge_continue.pt'
model_PLAN = load_model(load_model_name,model_PLAN)


In [149]:
torch.cuda.empty_cache()
torch.cuda.memory_allocated() / 1024**3

1.1263961791992188

# 4. Pred 

This function is corresponding with the continous workflow:
![示例图片](../assets/Continous_workflow.jpg)

In different region, you must change stack_value and value (Unfornately, it is a little trick and need more experience). For example, you can define P_stack_value by 0.15 times number of stations and define S_stack_value by 0.075 times number of stations.

In [135]:
str(start_time)

'2015-03-31 06:41:23.650000'

In [145]:
earthquake_time_list_p,earthquake_time_list_s,earthquake_time_list_total,earthquake_loc_list = pred(model_PLAN,
                                                                                                    ridge_loader,
                                                                                                    station_file_path,
                                                                                                    device,
                                                                                                    batch_start_time,
                                                                                                    P_stack_value = 2,
                                                                                                    S_stack_value = 1, 
                                                                                                    P_value = 0, 
                                                                                                    S_value = 0, 
                                                                                                    time_sample = 2000, 
                                                                                                    station_num = 2,
                                                                                                    time_start=str(start_time))

100%|███████████████████████████████████████████| 12/12 [00:00<00:00, 13.55it/s]


In [137]:
meta_tmp.source_origin_time

2595      2015-03-31 06:41:16.09
17788     2015-03-31 06:41:16.09
40398     2015-03-31 06:41:16.09
66580     2015-03-31 06:41:16.09
73756     2015-03-31 06:41:16.09
168218    2015-03-31 06:41:16.09
173921    2015-03-31 06:41:16.09
180792    2015-03-31 06:41:16.09
182545    2015-03-31 06:41:16.09
195679    2015-03-31 06:41:16.09
Name: source_origin_time, dtype: str

In [146]:
earthquake_time_list_total

['2015-03-31 06:41:24.126572',
 '2015-03-31 06:41:42.889443',
 '2015-03-31 06:41:48.449809']

After test, the break time calculate by s wave is more accuracy. Therefore, in this case, we utilized earthquake_time_list_s. For general, we can use the average time of P/S wave.

# 5. Save Catalog 

In [147]:
# delete duplicate event within 2s
merged_time_series, merged_coordinates = merge_time_series(earthquake_time_list_s, earthquake_loc_list,time_threshold = 2)
final_catalog = np.concatenate([np.array(merged_time_series).reshape(-1,1),merged_coordinates],axis=-1)
formatted_data = np.array([final_catalog[:,0]] + [final_catalog[:,i].astype(float).round(4) for i in range(1, final_catalog.shape[1])]).T
formatted_data = formatted_data[:,0:3]
# save catalog
np.savetxt('test_catalog.txt', formatted_data, fmt='%s', delimiter=', ')

In [148]:
formatted_data

array([['2015-03-31 06:41:23.294113', '118.9172', '32.9962'],
       ['2015-03-31 06:41:43.489673', '118.9694', '33.1062'],
       ['2015-03-31 06:41:49.484161', '118.9514', '32.9908']],
      dtype='<U32')